# Secondary evaluation track

This notebook runs the independent external-data checks for Labs 3–5. It does not modify the official Bayan datasets, labels, requirements or result files.

The first code cell creates the project folder and asks you to upload the three `secondary_*.py` scripts if they are missing.

In [ ]:
from pathlib import Path
import subprocess, sys

ROOT = Path('/content/BAYAN.DAICO-main')
SCRIPTS = ROOT / 'scripts'
SCRIPTS.mkdir(parents=True, exist_ok=True)
required = ['secondary_dataset_audit.py', 'secondary_topic_eval.py', 'secondary_retrieval_eval.py']
missing = [name for name in required if not (SCRIPTS / name).exists()]
if missing:
    print('Missing:', ', '.join(missing))
    print('Select the three secondary_*.py files from the file picker.')
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        (SCRIPTS / name).write_bytes(data)
missing = [name for name in required if not (SCRIPTS / name).exists()]
if missing:
    raise FileNotFoundError(f'Please upload these files: {missing}')
print('Project:', ROOT)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'scikit-learn'], check=True)
print('Dependencies are ready.')

## 1. Audit the external datasets

This checks split sizes, label coverage, Emirati/Gulf coverage, Arabic NER location spans and retrieval dialect coverage.

In [ ]:
subprocess.run([sys.executable, 'scripts/secondary_dataset_audit.py'], cwd=ROOT, check=True)

## 2. Run the independent Arabic topic baseline

This trains a separate 14-class TF-IDF baseline on ArBNTopic. Its score is independent evidence and is not substituted for Bayan's eight-label frozen-test score.

In [ ]:
subprocess.run([sys.executable, 'scripts/secondary_topic_eval.py'], cwd=ROOT, check=True)

## 3. Run the independent retrieval benchmark

ArabicRAGB supplies an explicit positive passage for each query, so recall@10 and MRR@10 are defined without reusing Bayan's cyclic case IDs.

In [ ]:
subprocess.run([sys.executable, 'scripts/secondary_retrieval_eval.py'], cwd=ROOT, check=True)

In [ ]:
import json
for name in ['dataset_audit.json', 'arbn_topic_eval.json', 'arabicragb_retrieval.json']:
    path = ROOT / 'artifacts' / 'secondary' / name
    print(f'\n{name}')
    print(json.dumps(json.loads(path.read_text()), ensure_ascii=False, indent=2))